In [ ]:
import mysql.connector
from mysql.connector import Error

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()


True

CREATE USER 'yt_user'@'localhost' IDENTIFIED BY XXXXXXXXXX(password);
GRANT ALL PRIVILEGES ON youtube_analytics.* TO 'yt_user'@'localhost';
FLUSH PRIVILEGES;            


ensure to run this command so that it will allocate the permissions

In [ ]:
try:
    connection=mysql.connector.connect(
        host='localhost',
        user="yt_user",
        password=os.getenv("MYSQL_PASSWORD"),
        database='youtube_analytics'
    )
    if connection.is_connected():
        print("Successfully connected to MySQL Platform")
except Error as e:
    print(f"Error connecting to MySQL Platform: {e}")

Successfully connected to MySQL Platform


In [ ]:
def insert_channel(connection, channel_data):
    cursor = connection.cursor()

    query = """
    INSERT INTO channels (channel_id, channel_name, subscribers, total_videos, total_views, category)
    VALUES (%s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        subscribers = VALUES(subscribers),
        total_videos = VALUES(total_videos),
        total_views = VALUES(total_views),
        category = VALUES(category)
    """

    values = (
        channel_data["channel_id"],
        channel_data["channel_name"],
        channel_data["subscribers"],
        channel_data["total_videos"],
        channel_data["total_views"],
        channel_data["category"]
    )

    cursor.execute(query, values)
    connection.commit()
    cursor.close()


In [ ]:
def insert_videos(connection, videos_list):
    cursor = connection.cursor()

    query = """
    INSERT INTO videos (video_id, channel_id, title, published_date, views, likes, comments)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        views = VALUES(views),
        likes = VALUES(likes),
        comments = VALUES(comments)
    """

    data = [
        (
            video["video_id"],
            video["channel_id"],
            video["title"],
            video["published_date"],
            video["views"],
            video["likes"],
            video["comments"]
        )
        for video in videos_list
    ]

    cursor.executemany(query, data)
    connection.commit()
    cursor.close()


In [ ]:
channel_data = {
    "channel_id": "UC_x5XG1OV2P6uZZ5FSM9Ttw",
    "channel_name": "Google Developers",
    "subscribers": 2600000,
    "total_videos": 6000,
    "total_views": 200000000,
    "category": "big"
}

insert_channel(connection, channel_data)


In [ ]:
videos_list = [
    {
        "video_id": "abc123",
        "channel_id": "UC_x5XG1OV2P6uZZ5FSM9Ttw",
        "title": "Sample Video",
        "published_date": "2024-01-01 10:00:00",
        "views": 100000,
        "likes": 5000,
        "comments": 300
    }
]

insert_videos(connection, videos_list)
